# Prometheus Data Fetcher for Training

This notebook demonstrates how to fetch real data from various APIs for model training.

In [ ]:
# Import required libraries
import os
import sys
import requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import json
from typing import Dict, List, Tuple
import time

# Add current directory to path to import api_config
sys.path.append('.')

# Import API configuration
from api_config import APICredentials, APIEndpoints, get_auth_headers, get_sentinel_hub_token

## 1. USGS Earthquake Data Fetcher

In [ ]:
class USGSDataFetcher:
    def __init__(self):
        self.base_url = APIEndpoints.USGS_EARTHQUAKE_QUERY
        self.headers = get_auth_headers("usgs")
    
    def fetch_earthquakes(self, 
                         start_date: str, 
                         end_date: str, 
                         min_magnitude: float = 2.5,
                         max_results: int = 10000) -> pd.DataFrame:
        """
        Fetch earthquake data from USGS
        """
        params = {
            "format": "geojson",
            "starttime": start_date,
            "endtime": end_date,
            "minmagnitude": min_magnitude,
            "limit": max_results,
            "orderby": "time"
        }
        
        response = requests.get(self.base_url, params=params, headers=self.headers)
        
        if response.status_code == 200:
            data = response.json()
            
            # Extract features
            earthquakes = []
            for feature in data['features']:
                props = feature['properties']
                coords = feature['geometry']['coordinates']
                
                earthquakes.append({
                    'time': pd.to_datetime(props['time'], unit='ms'),
                    'latitude': coords[1],
                    'longitude': coords[0],
                    'depth': coords[2],
                    'magnitude': props['mag'],
                    'place': props['place'],
                    'type': props['type'],
                    'status': props['status']
                })
            
            return pd.DataFrame(earthquakes)
        else:
            raise Exception(f"Failed to fetch data: {response.status_code}")
    
    def fetch_waveform_ids(self, event_id: str) -> List[str]:
        """
        Get waveform IDs for a specific earthquake event
        """
        url = f"https://earthquake.usgs.gov/fdsnws/event/1/query?eventid={event_id}&format=geojson"
        response = requests.get(url, headers=self.headers)
        
        if response.status_code == 200:
            data = response.json()
            # Extract associated waveform IDs if available
            return data.get('properties', {}).get('ids', '').split(',')
        return []

# Test USGS fetcher
usgs_fetcher = USGSDataFetcher()
end_date = datetime.now()
start_date = end_date - timedelta(days=7)

print("Fetching earthquake data from last 7 days...")
earthquakes_df = usgs_fetcher.fetch_earthquakes(
    start_date.isoformat(),
    end_date.isoformat(),
    min_magnitude=4.0
)

print(f"Found {len(earthquakes_df)} earthquakes")
earthquakes_df.head()

## 2. IRIS Seismic Waveform Fetcher

In [ ]:
class IRISDataFetcher:
    def __init__(self):
        self.creds = APICredentials()
        self.base_url = APIEndpoints.IRIS_BASE
    
    def fetch_stations(self, 
                      latitude: float, 
                      longitude: float, 
                      radius: float = 10.0) -> pd.DataFrame:
        """
        Fetch seismic stations within radius of a point
        """
        params = {
            "latitude": latitude,
            "longitude": longitude,
            "maxradius": radius,
            "format": "text",
            "level": "station"
        }
        
        url = f"{APIEndpoints.IRIS_STATION}/query"
        response = requests.get(url, params=params)
        
        if response.status_code == 200:
            # Parse text response
            lines = response.text.strip().split('\n')
            if len(lines) > 1:
                # Skip header
                stations = []
                for line in lines[1:]:
                    parts = line.split('|')
                    if len(parts) >= 6:
                        stations.append({
                            'network': parts[0],
                            'station': parts[1],
                            'latitude': float(parts[2]),
                            'longitude': float(parts[3]),
                            'elevation': float(parts[4]),
                            'name': parts[5] if len(parts) > 5 else ''
                        })
                return pd.DataFrame(stations)
        return pd.DataFrame()
    
    def build_waveform_url(self, 
                          network: str, 
                          station: str, 
                          start_time: str, 
                          end_time: str) -> str:
        """
        Build URL for waveform data request
        """
        params = {
            "network": network,
            "station": station,
            "location": "*",
            "channel": "BH*",  # Broadband high-gain
            "starttime": start_time,
            "endtime": end_time,
            "format": "miniseed"
        }
        
        param_str = "&".join([f"{k}={v}" for k, v in params.items()])
        return f"{APIEndpoints.IRIS_DATASELECT}/query?{param_str}"

# Test IRIS fetcher
iris_fetcher = IRISDataFetcher()

# Find stations near a recent earthquake
if len(earthquakes_df) > 0:
    eq = earthquakes_df.iloc[0]
    print(f"Finding stations near earthquake at {eq['latitude']}, {eq['longitude']}")
    
    stations_df = iris_fetcher.fetch_stations(eq['latitude'], eq['longitude'], radius=5.0)
    print(f"Found {len(stations_df)} stations")
    stations_df.head()

## 3. Sentinel Hub Thermal Data Fetcher

In [ ]:
class SentinelHubFetcher:
    def __init__(self):
        self.token = None
        self.token_expiry = None
    
    def _get_token(self):
        """Get or refresh OAuth token"""
        if self.token is None or datetime.now() >= self.token_expiry:
            self.token = get_sentinel_hub_token()
            self.token_expiry = datetime.now() + timedelta(hours=1)
        return self.token
    
    def search_thermal_data(self, 
                           bbox: List[float], 
                           start_date: str, 
                           end_date: str,
                           max_cloud_cover: int = 30) -> List[Dict]:
        """
        Search for thermal satellite data
        bbox: [min_lon, min_lat, max_lon, max_lat]
        """
        token = self._get_token()
        
        headers = {
            "Authorization": f"Bearer {token}",
            "Content-Type": "application/json"
        }
        
        # Search request for Sentinel-3 SLSTR (thermal)
        search_request = {
            "bbox": bbox,
            "datetime": f"{start_date}T00:00:00Z/{end_date}T23:59:59Z",
            "collections": ["sentinel-3-slstr"],
            "limit": 100,
            "query": {
                "eo:cloud_cover": {
                    "lt": max_cloud_cover
                }
            }
        }
        
        response = requests.post(
            APIEndpoints.SENTINEL_HUB_CATALOG,
            json=search_request,
            headers=headers
        )
        
        if response.status_code == 200:
            return response.json().get('features', [])
        else:
            print(f"Error: {response.status_code} - {response.text}")
            return []
    
    def create_thermal_evalscript(self) -> str:
        """
        Create evalscript for thermal band processing
        """
        return """
        //VERSION=3
        function setup() {
            return {
                input: ["S8", "S9", "F1"],  // Thermal bands
                output: {
                    bands: 3,
                    sampleType: "FLOAT32"
                }
            };
        }
        
        function evaluatePixel(sample) {
            // Convert to temperature in Celsius
            let bt8 = sample.S8 - 273.15;
            let bt9 = sample.S9 - 273.15;
            let fire = sample.F1;
            
            return [bt8, bt9, fire];
        }
        """

# Test Sentinel Hub fetcher
sentinel_fetcher = SentinelHubFetcher()

# Search for thermal data around a volcano (example: Mount Etna)
etna_bbox = [14.9, 37.7, 15.1, 37.8]  # Sicily, Italy
end_date = datetime.now()
start_date = end_date - timedelta(days=30)

print("Searching for thermal data around Mount Etna...")
thermal_scenes = sentinel_fetcher.search_thermal_data(
    etna_bbox,
    start_date.strftime('%Y-%m-%d'),
    end_date.strftime('%Y-%m-%d')
)

print(f"Found {len(thermal_scenes)} thermal scenes")
if thermal_scenes:
    print(f"Latest scene: {thermal_scenes[0]['properties']['datetime']}")

## 4. NASA EarthData InSAR Fetcher

In [ ]:
class NASAEarthDataFetcher:
    def __init__(self):
        self.headers = get_auth_headers("earthdata")
    
    def search_insar_data(self, 
                         bbox: List[float], 
                         start_date: str, 
                         end_date: str) -> List[Dict]:
        """
        Search for InSAR/deformation data products
        """
        # CMR API search
        params = {
            "short_name": "ARIA_S1_GUNW",  # ARIA Sentinel-1 products
            "bounding_box": ",".join(map(str, bbox)),
            "temporal": f"{start_date}T00:00:00Z,{end_date}T23:59:59Z",
            "page_size": 100,
            "pretty": "true"
        }
        
        url = f"{APIEndpoints.EARTHDATA_CMR}/search/granules.json"
        response = requests.get(url, params=params, headers=self.headers)
        
        if response.status_code == 200:
            data = response.json()
            return data.get('feed', {}).get('entry', [])
        else:
            print(f"Error: {response.status_code}")
            return []
    
    def get_data_urls(self, granule: Dict) -> List[str]:
        """
        Extract data URLs from granule metadata
        """
        urls = []
        for link in granule.get('links', []):
            if link.get('rel') == 'http://esipfed.org/ns/fedsearch/1.1/data#':
                urls.append(link['href'])
        return urls

# Test NASA EarthData fetcher
nasa_fetcher = NASAEarthDataFetcher()

# Search for InSAR data (example: California)
ca_bbox = [-124.0, 32.0, -114.0, 42.0]
print("Searching for InSAR data over California...")

insar_products = nasa_fetcher.search_insar_data(
    ca_bbox,
    (datetime.now() - timedelta(days=30)).strftime('%Y-%m-%d'),
    datetime.now().strftime('%Y-%m-%d')
)

print(f"Found {len(insar_products)} InSAR products")

## 5. Combined Data Pipeline for Training

In [ ]:
class PrometheusDataPipeline:
    """
    Combined data pipeline for fetching and preparing training data
    """
    def __init__(self):
        self.usgs = USGSDataFetcher()
        self.iris = IRISDataFetcher()
        self.sentinel = SentinelHubFetcher()
        self.nasa = NASAEarthDataFetcher()
    
    def fetch_training_data_for_region(self, 
                                      region_name: str,
                                      bbox: List[float],
                                      start_date: str,
                                      end_date: str,
                                      output_dir: str = "./training_data"):
        """
        Fetch all available data for a region and time period
        """
        os.makedirs(output_dir, exist_ok=True)
        region_dir = os.path.join(output_dir, region_name)
        os.makedirs(region_dir, exist_ok=True)
        
        # 1. Fetch earthquake catalog
        print(f"Fetching earthquakes for {region_name}...")
        earthquakes = self.usgs.fetch_earthquakes(start_date, end_date, min_magnitude=2.5)
        
        # Filter by bbox
        mask = (
            (earthquakes['longitude'] >= bbox[0]) & 
            (earthquakes['longitude'] <= bbox[2]) &
            (earthquakes['latitude'] >= bbox[1]) & 
            (earthquakes['latitude'] <= bbox[3])
        )
        earthquakes_region = earthquakes[mask]
        
        # Save earthquake catalog
        earthquakes_region.to_csv(
            os.path.join(region_dir, 'earthquakes.csv'), 
            index=False
        )
        print(f"Saved {len(earthquakes_region)} earthquakes")
        
        # 2. Fetch station information
        print(f"Fetching seismic stations...")
        center_lat = (bbox[1] + bbox[3]) / 2
        center_lon = (bbox[0] + bbox[2]) / 2
        stations = self.iris.fetch_stations(center_lat, center_lon, radius=5.0)
        
        if len(stations) > 0:
            stations.to_csv(
                os.path.join(region_dir, 'stations.csv'),
                index=False
            )
            print(f"Saved {len(stations)} stations")
        
        # 3. Search for thermal imagery
        print(f"Searching for thermal imagery...")
        thermal_scenes = self.sentinel.search_thermal_data(bbox, start_date, end_date)
        
        if thermal_scenes:
            with open(os.path.join(region_dir, 'thermal_scenes.json'), 'w') as f:
                json.dump(thermal_scenes, f, indent=2)
            print(f"Found {len(thermal_scenes)} thermal scenes")
        
        # 4. Search for InSAR data
        print(f"Searching for InSAR products...")
        insar_products = self.nasa.search_insar_data(bbox, start_date, end_date)
        
        if insar_products:
            with open(os.path.join(region_dir, 'insar_products.json'), 'w') as f:
                json.dump(insar_products, f, indent=2)
            print(f"Found {len(insar_products)} InSAR products")
        
        # Create metadata summary
        metadata = {
            "region": region_name,
            "bbox": bbox,
            "time_range": [start_date, end_date],
            "data_counts": {
                "earthquakes": len(earthquakes_region),
                "stations": len(stations),
                "thermal_scenes": len(thermal_scenes),
                "insar_products": len(insar_products)
            },
            "created_at": datetime.now().isoformat()
        }
        
        with open(os.path.join(region_dir, 'metadata.json'), 'w') as f:
            json.dump(metadata, f, indent=2)
        
        print(f"\nData collection complete for {region_name}")
        return metadata

# Example: Fetch data for multiple volcanic regions
pipeline = PrometheusDataPipeline()

# Define regions of interest
regions = [
    {
        "name": "mount_etna",
        "bbox": [14.9, 37.7, 15.1, 37.8]  # Sicily, Italy
    },
    {
        "name": "hawaii_kilauea",
        "bbox": [-155.4, 19.3, -155.2, 19.5]  # Hawaii, USA
    },
    {
        "name": "japan_fuji",
        "bbox": [138.6, 35.2, 138.8, 35.4]  # Japan
    }
]

# Fetch data for each region
end_date = datetime.now().strftime('%Y-%m-%d')
start_date = (datetime.now() - timedelta(days=90)).strftime('%Y-%m-%d')

for region in regions[:1]:  # Process first region as example
    print(f"\n{'='*60}")
    print(f"Processing {region['name']}...")
    print(f"{'='*60}")
    
    metadata = pipeline.fetch_training_data_for_region(
        region['name'],
        region['bbox'],
        start_date,
        end_date
    )
    
    time.sleep(2)  # Be nice to APIs

## 6. Data Download URLs Generator

In [ ]:
def generate_download_script(region_dir: str):
    """
    Generate wget/curl commands for downloading actual data files
    """
    script_lines = ["#!/bin/bash", "", "# Prometheus data download script", ""]
    
    # Read metadata
    with open(os.path.join(region_dir, 'metadata.json'), 'r') as f:
        metadata = json.load(f)
    
    script_lines.append(f"# Region: {metadata['region']}")
    script_lines.append(f"# Time range: {metadata['time_range'][0]} to {metadata['time_range'][1]}")
    script_lines.append("")
    
    # Add download commands for each data type
    script_lines.append("# Create directories")
    script_lines.append("mkdir -p waveforms thermal insar")
    script_lines.append("")
    
    # IRIS waveform URLs (example)
    if os.path.exists(os.path.join(region_dir, 'stations.csv')):
        stations_df = pd.read_csv(os.path.join(region_dir, 'stations.csv'))
        if len(stations_df) > 0:
            script_lines.append("# IRIS Waveform data")
            iris = IRISDataFetcher()
            
            for idx, station in stations_df.head(5).iterrows():  # First 5 stations
                url = iris.build_waveform_url(
                    station['network'],
                    station['station'],
                    metadata['time_range'][0],
                    metadata['time_range'][1]
                )
                filename = f"waveforms/{station['network']}_{station['station']}.mseed"
                script_lines.append(f"wget -O {filename} \"{url}\"")
            script_lines.append("")
    
    # Save download script
    script_path = os.path.join(region_dir, 'download_data.sh')
    with open(script_path, 'w') as f:
        f.write('\n'.join(script_lines))
    
    print(f"Download script saved to: {script_path}")
    return script_path

# Generate download script for the first region
if regions:
    region_dir = os.path.join('./training_data', regions[0]['name'])
    if os.path.exists(region_dir):
        generate_download_script(region_dir)